# RouteAlert — เทรนโมเดลตรวจอุบัติเหตุ v2 (แก้ปัญหา "หน้าคน = อุบัติเหตุ")

**ทำไมต้องมีเวอร์ชันนี้:** โมเดลรุ่นก่อน (EfficientNetV2-L) แม่น 100% กับรูปใน dataset แต่พอเจอรูปคน/เซลฟี่
ในแอปจริงกลับตอบว่า "อุบัติเหตุ" (0.6–0.9) เพราะโฟลเดอร์ `non_accident` มี **แต่รถยนต์สภาพดี** — โมเดลจึงเรียนรู้แค่
"รถสภาพดี = ปกติ อย่างอื่นทั้งหมด = อุบัติเหตุ"

**สิ่งที่เปลี่ยน**
1. **เพิ่มรูป non_accident ที่หลากหลายอัตโนมัติ** — หน้าคน (LFW) + สิ่งของ/ยานพาหนะสภาพปกติ 100 หมวด (Caltech-101 รวมมอเตอร์ไซค์) จาก TensorFlow Datasets (ฟรี ไม่ต้องสมัคร)
2. **แยกรูปหน้าคนชุดหนึ่งไว้ทดสอบโดยเฉพาะ** — ยืนยันว่าแก้ปัญหาเดิมได้จริง (ไม่ใช่แค่แม่นกับรูปใน dataset)
3. **โมเดลเล็กลง EfficientNetV2-B0 ที่ 224px** — ไฟล์ ~7 MB (เดิม 470 MB) เร็วบนมือถือกว่ามาก ความแม่นยำสำหรับงาน 2 คลาสใกล้เคียงกัน
4. โมเดลรับภาพดิบ 0–255 (มีขั้นปรับค่าสีในตัว) ตรงกับที่แอป Flutter ส่ง — ใช้กับ `accident_image_classifier_service.dart` ได้ทันทีโดยไม่ต้องแก้โค้ดแอป

**ก่อนรัน**
- Runtime → Change runtime type → **T4 GPU**
- เตรียม `dataset.zip` ตัวเดิม (มีโฟลเดอร์ `accident/` และ `non_accident/`)
- ถ้ามีเวลา: ใส่รูปถ่ายเองเพิ่มใน `non_accident/` ก่อน zip เช่น เซลฟี่, ถนนปกติ, มอเตอร์ไซค์ที่จอดอยู่, ในห้อง — ช่วยให้แม่นกับรูปจากมือถือจริงที่สุด
- ใช้เวลาประมาณ 30–60 นาที

In [ ]:
# ==== 1. เตรียมสภาพแวดล้อม ====
# Colab มี TensorFlow อยู่แล้ว — ติดตั้งเพิ่มเฉพาะที่ขาด (อย่าติดตั้ง tensorflow ทับ จะทำให้ GPU ใช้ไม่ได้)
!pip install -q tensorflow-datasets importlib_resources

import tensorflow as tf
import numpy as np
import random, os, gc, pathlib, shutil
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("✅ GPU:", gpus[0].name if gpus else "⚠️ ไม่พบ GPU — Runtime > Change runtime type > T4 GPU")
tf.random.set_seed(42); np.random.seed(42); random.seed(42)

In [ ]:
# ==== 2. นำเข้า dataset.zip ====
import zipfile

shutil.rmtree("dataset", ignore_errors=True)
shutil.rmtree("_extract", ignore_errors=True)

USE_GOOGLE_DRIVE = False  # True ถ้าวาง dataset.zip ไว้ที่ MyDrive/dataset.zip

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = "/content/drive/MyDrive/dataset.zip"
else:
    from google.colab import files
    print("เลือกไฟล์ dataset.zip:")
    uploaded = files.upload()
    zip_path = next(iter(uploaded))

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("_extract")
for junk in pathlib.Path("_extract").rglob("__MACOSX"):
    shutil.rmtree(junk, ignore_errors=True)

found = [d for d in pathlib.Path("_extract").rglob("accident")
         if d.is_dir() and (d.parent / "non_accident").is_dir()]
assert found, "❌ ไม่พบโฟลเดอร์ accident/ กับ non_accident/ ในไฟล์ zip"
pathlib.Path("dataset").mkdir()
for name in ["accident", "non_accident"]:
    shutil.move(str(found[0].parent / name), f"dataset/{name}")
shutil.rmtree("_extract", ignore_errors=True)

for c in ["accident", "non_accident"]:
    print(f"{c}: {len(list(pathlib.Path('dataset', c).glob('*')))} รูป")

In [ ]:
# ==== 3. เพิ่มรูป non_accident ที่หลากหลาย (แก้ปัญหาหน้าคน = อุบัติเหตุ) ====
# เดิม non_accident มีแต่รถยนต์สภาพดี — เพิ่มหน้าคนและสิ่งของทั่วไปเข้าไป และแยกหน้าคนชุดหนึ่ง
# ไว้ทดสอบตอนท้าย (people_check/) ซึ่งโมเดลจะไม่เคยเห็นตอนเทรน
import tensorflow_datasets as tfds
from PIL import Image

N_FACES_TRAIN = 350     # หน้าคนสำหรับเทรน
N_FACES_CHECK = 80      # หน้าคนสำหรับทดสอบตอนท้าย (ไม่ใช้เทรน)
N_OBJECTS = 400         # สิ่งของ/ยานพาหนะสภาพปกติ

non_dir = pathlib.Path("dataset/non_accident")
check_dir = pathlib.Path("people_check"); shutil.rmtree(check_dir, ignore_errors=True); check_dir.mkdir()

def save_examples(ds, n, prefix, out_dir, skip_labels=()):
    saved = 0
    for ex in ds:
        if saved >= n: break
        if skip_labels and int(ex["label"]) in skip_labels: continue
        img = ex["image"].numpy()
        if img.ndim == 2 or img.shape[-1] == 1:
            img = np.repeat(img.reshape(img.shape[0], img.shape[1], 1), 3, axis=-1)
        Image.fromarray(img).convert("RGB").save(out_dir / f"{prefix}_{saved:04d}.jpg", quality=92)
        saved += 1
    return saved

added = {}
try:
    faces = tfds.load("lfw", split="train", shuffle_files=True).shuffle(5000, seed=42)
    added["faces_train"] = save_examples(faces.take(N_FACES_TRAIN), N_FACES_TRAIN, "extra_face", non_dir)
    added["faces_check"] = save_examples(faces.skip(N_FACES_TRAIN).take(N_FACES_CHECK), N_FACES_CHECK, "face", check_dir)
except Exception as e:
    print("⚠️ ดาวน์โหลดชุดหน้าคน (LFW) ไม่สำเร็จ:", e)
    print("   → ใส่รูปเซลฟี่/รูปคนเองลงใน dataset/non_accident และ people_check/ แทนได้")

try:
    caltech, info = tfds.load("caltech101", split="train", shuffle_files=True, with_info=True)
    names = info.features["label"].names
    # ข้ามหมวดที่ไม่ใช่ภาพจริง/ซ้ำกับหน้าคนที่เพิ่มแล้ว
    skip = {names.index(n) for n in ["BACKGROUND_Google", "Faces", "Faces_easy"] if n in names}
    added["objects"] = save_examples(caltech.shuffle(4000, seed=42), N_OBJECTS, "extra_obj", non_dir, skip)
except Exception as e:
    print("⚠️ ดาวน์โหลด Caltech-101 ไม่สำเร็จ:", e)

print("เพิ่มรูปแล้ว:", added)
for c in ["accident", "non_accident"]:
    print(f"{c}: {len(list(pathlib.Path('dataset', c).glob('*')))} รูป")
print(f"people_check: {len(list(check_dir.glob('*')))} รูป (ใช้ทดสอบเท่านั้น)")

In [ ]:
# ==== 4. กรองไฟล์ภาพเสีย/เปิดไม่ได้ ====
def scrub(root):
    removed = 0
    for p in pathlib.Path(root).rglob("*"):
        if not p.is_file(): continue
        try:
            with Image.open(p) as im: im.verify()
            with Image.open(p) as im: im.convert("RGB")
        except Exception:
            p.unlink(); removed += 1
    return removed

print("ลบไฟล์เสีย:", scrub("dataset") + scrub("people_check"))

In [ ]:
# ==== 5. แบ่ง Train 80% / Val 10% / Test 10% และเตรียมข้อมูล (224x224) ====
from sklearn.model_selection import train_test_split

IMG_SIZE = 224
BATCH_SIZE = 32
class_names = ["accident", "non_accident"]   # ลำดับนี้ = ลำดับใน accident_labels.txt

files_all, labels_all = [], []
for c in class_names:
    for f in sorted(pathlib.Path("dataset", c).glob("*")):
        if f.is_file() and not f.name.startswith("."):
            files_all.append(f); labels_all.append(c)

tr_f, tmp_f, tr_l, tmp_l = train_test_split(files_all, labels_all, test_size=0.2, random_state=42, stratify=labels_all)
va_f, te_f, va_l, te_l = train_test_split(tmp_f, tmp_l, test_size=0.5, random_state=42, stratify=tmp_l)

split_root = pathlib.Path("split"); shutil.rmtree(split_root, ignore_errors=True)
for name, fs, ls in [("train", tr_f, tr_l), ("val", va_f, va_l), ("test", te_f, te_l)]:
    for c in class_names: (split_root / name / c).mkdir(parents=True)
    for f, l in zip(fs, ls): shutil.copy(f, split_root / name / l / f.name)
    print(f"{name}: {len(fs)} รูป")

def load(name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        split_root / name, label_mode="categorical", class_names=class_names,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=shuffle, seed=42)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = load("train", True).prefetch(AUTOTUNE)
val_ds = load("val", False).prefetch(AUTOTUNE)
test_ds = load("test", False).prefetch(AUTOTUNE)

# กันข้อมูลไม่สมดุล (non_accident เยอะกว่าหลังเพิ่มรูป)
n_acc = sum(1 for l in tr_l if l == "accident"); n_non = len(tr_l) - n_acc
class_weight = {0: len(tr_l) / (2 * n_acc), 1: len(tr_l) / (2 * n_non)}
print("class_weight:", class_weight)

# จำลองรูปถ่ายจากมือถือจริง: มุมเอียง ซูม แสงไม่คงที่
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
], name="augment")

In [ ]:
# ==== 6. สร้างโมเดล EfficientNetV2-B0 ====
# include_preprocessing=True → โมเดลรับภาพดิบ 0-255 แล้วปรับค่าสีเอง ตรงกับที่แอปส่ง
base_model = tf.keras.applications.EfficientNetV2B0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
    weights="imagenet", include_preprocessing=True)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax", name="prediction")(x)
model = tf.keras.Model(inputs, outputs, name="routealert_accident_b0")

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
print(f"พารามิเตอร์ทั้งหมด: {model.count_params():,}")

In [ ]:
# ==== 7. เทรน 2 ขั้น: หัวโมเดลก่อน แล้วค่อยปลดล็อกชั้นบนของ backbone ====
ckpt = tf.keras.callbacks.ModelCheckpoint("best.keras", monitor="val_accuracy", save_best_only=True, mode="max")

# Phase 1 — เทรนเฉพาะหัว (backbone ล็อกไว้)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=class_weight,
               callbacks=[ckpt, tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)])

# Phase 2 — fine-tune ครึ่งบนของ backbone (BatchNorm ล็อกไว้เสมอ กันพังตอน batch เล็ก)
base_model.trainable = True
cut = int(len(base_model.layers) * 0.5)
for layer in base_model.layers[:cut]: layer.trainable = False
for layer in base_model.layers[cut:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization): layer.trainable = False

start = len(h1.history["loss"])
steps = len(train_ds) * 40
model.compile(optimizer=tf.keras.optimizers.Adam(tf.keras.optimizers.schedules.CosineDecay(3e-5, steps, alpha=0.05)),
              loss=loss_fn, metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, initial_epoch=start, epochs=start + 40, class_weight=class_weight,
               callbacks=[ckpt, tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True)])

model = tf.keras.models.load_model("best.keras")
print("✅ โหลดโมเดลที่ดีที่สุดจาก val_accuracy แล้ว")

import matplotlib.pyplot as plt
acc = h1.history["accuracy"] + h2.history["accuracy"]; val = h1.history["val_accuracy"] + h2.history["val_accuracy"]
plt.plot(acc, label="train"); plt.plot(val, label="val"); plt.axvline(start - 0.5, ls="--", c="gray")
plt.title("Accuracy (เส้นประ = เริ่ม fine-tune)"); plt.legend(); plt.show()

In [ ]:
# ==== 8. ประเมินผลบน Test Set (รูปที่โมเดลไม่เคยเห็น) ====
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, labels in test_ds:
    y_true += list(np.argmax(labels.numpy(), axis=1))
    y_pred += list(np.argmax(model.predict(images, verbose=0), axis=1))
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print("Confusion matrix (แถว = จริง, คอลัมน์ = ทำนาย):")
print(confusion_matrix(y_true, y_pred))

In [ ]:
# ==== 9. ทดสอบปัญหาเดิม: หน้าคนต้องไม่ใช่อุบัติเหตุ ====
# people_check/ ไม่ได้ถูกใช้เทรนเลย — ถ้าผ่าน แปลว่าแก้ปัญหา "หน้าคน = อุบัติเหตุ" ได้จริง
import matplotlib.pyplot as plt
check_files = sorted(pathlib.Path("people_check").glob("*"))
if check_files:
    wrong = 0; worst = []
    for f in check_files:
        img = np.asarray(Image.open(f).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)[None]
        p_acc = float(model.predict(img, verbose=0)[0][0])
        if p_acc >= 0.5: wrong += 1
        worst.append((p_acc, f))
    ok = len(check_files) - wrong
    print(f"หน้าคน {len(check_files)} รูป → ตอบถูกว่าไม่ใช่อุบัติเหตุ {ok} รูป ({ok / len(check_files) * 100:.1f}%)")
    print("✅ ผ่าน" if ok / len(check_files) >= 0.95 else "⚠️ ยังพลาดเยอะ — เพิ่มรูปคน/เซลฟี่ใน non_accident แล้วเทรนใหม่")
    worst.sort(reverse=True)
    plt.figure(figsize=(12, 3))
    for k, (p, f) in enumerate(worst[:6]):
        plt.subplot(1, 6, k + 1); plt.imshow(Image.open(f)); plt.axis("off"); plt.title(f"P(อุบัติเหตุ)={p:.2f}", fontsize=9)
    plt.suptitle("หน้าคนที่โมเดลคิดว่าคล้ายอุบัติเหตุที่สุด"); plt.show()
else:
    print("ไม่มีรูปใน people_check/ — ข้ามขั้นนี้")

# (ไม่บังคับ) อัปโหลดรูปของคุณเองมาลองทาย เช่น เซลฟี่ รูปถนน รูปรถชน
TRY_MY_PHOTOS = False
if TRY_MY_PHOTOS:
    from google.colab import files
    for name, data in files.upload().items():
        import io
        img = np.asarray(Image.open(io.BytesIO(data)).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)[None]
        p = model.predict(img, verbose=0)[0]
        print(f"{name}: อุบัติเหตุ {p[0] * 100:.1f}% / ไม่ใช่ {p[1] * 100:.1f}%")

In [ ]:
# ==== 10. แปลงเป็น TFLite (float32 input/output ตามที่แอปใช้) ====
# ใช้ dynamic-range quantization (บีบเฉพาะน้ำหนัก) — ห้ามทำ full integer เพราะแอปส่ง/อ่านค่าเป็น float
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open("accident_classifier.tflite", "wb") as f: f.write(tflite_model)
with open("accident_labels.txt", "w") as f:
    for name in class_names: f.write(name + "\n")

interpreter = tf.lite.Interpreter(model_path="accident_classifier.tflite"); interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]; out = interpreter.get_output_details()[0]
print(f"📦 ขนาดไฟล์: {len(tflite_model) / 1024 / 1024:.1f} MB")
print("input:", inp["shape"], inp["dtype"].__name__, "| output:", out["shape"], out["dtype"].__name__)
assert inp["dtype"] == np.float32 and out["dtype"] == np.float32, "input/output ต้องเป็น float32"

In [ ]:
# ==== 11. ตรวจว่าไฟล์ TFLite ให้ผลตรงกับโมเดลต้นฉบับ ====
# เทียบความน่าจะเป็นโดยตรง — การบีบไฟล์ทำให้ค่าต่างกันเล็กน้อยเป็นปกติ (ไม่เกิน ~0.05)
images, _ = next(iter(test_ds))
diffs = []
for k in range(len(images)):
    x = images[k:k + 1].numpy().astype(np.float32)   # ภาพดิบ 0-255 แบบเดียวกับที่แอปส่ง
    keras_p = model.predict(x, verbose=0)[0]
    interpreter.set_tensor(inp["index"], x); interpreter.invoke()
    diffs.append(float(np.max(np.abs(keras_p - interpreter.get_tensor(out["index"])[0]))))
print(f"ความต่างของความน่าจะเป็นสูงสุด {max(diffs):.3f} (เฉลี่ย {np.mean(diffs):.3f}) จาก {len(images)} รูป")
print("✅ ไฟล์ TFLite ใช้ได้" if max(diffs) < 0.08 else "⚠️ ต่างกันมากเกินไป ตรวจสอบการแปลงอีกครั้ง")

In [ ]:
# ==== 12. ดาวน์โหลดไฟล์ไปใส่แอป ====
# วางทั้ง 2 ไฟล์ทับของเดิมใน route-alert-app/assets/models/ แล้วรันแอปใหม่ทั้งแอป (hot reload ไม่พอ)
from google.colab import files
files.download("accident_classifier.tflite")
files.download("accident_labels.txt")